# Talking to Foundation Models: Closed APIs vs. Open Weights
## Hands-On Session

<!-- Customize this header with your own course name, module number, and instructor details. -->

Welcome! Today we'll get a large language model to do work for us **two completely different ways**:

1. **A closed-source foundation model API** — [Google Gemini](https://ai.google.dev/gemini-api/docs). You'll send a request over the internet to a model running on Google's servers and get a response back. No download, no GPU required — but you don't control the weights, and every request leaves your machine.
2. **An open-weight model, running locally** — [Google Gemma 3](https://huggingface.co/google/gemma-3-1b-it), the open-weight sibling of Gemini. You'll download the actual model weights onto this Colab machine and run inference yourself, on a GPU Google is lending you for free.

By the end, we'll compare both approaches head-to-head on the exact same prompt (in terms of speed, cost, setup effort, and what you give up or gain).

### Before you start

- Select a GPU runtime: **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**, then click `Save`. Part 2 (the local open-weight model) needs it; Part 1 (the API) doesn't, but let's have it ready from the start.
- You'll need two free accounts. Both take under a minute to set up, and we'll walk through each right when it's needed:
  - A **Google AI Studio** account, for a free Gemini API key.
  - A **Hugging Face** account, to download the open-weight model.

### Agenda (~60 minutes)
| Section | Time |
|---|---|
| 0. Setup & Orientation | 5 min |
| 1. Part 1 — The Gemini API (closed-source) | 20 min |
| 2. Part 2 — Gemma 3 running locally (open-weight) | 20 min |
| 3. Head-to-Head Comparison | 8 min |
| 4. Bonus — Swap in a different open-weight model | optional |
| Wrap-Up | 2 min |

Just run each cell in order — every section explains what it's doing and why before you get to the code.

---
# 0. Setup & Orientation

In this section we will:
- Install the libraries we need for both parts of today's session
- Confirm we have a GPU available (needed for Part 2)
- Walk through the core idea we're exploring today: two different ways to "get" a large language model to work for you


In [ ]:
# Install/upgrade the libraries used in this session.
# - google-genai:    the official Python SDK for the Gemini API (Part 1)
# - transformers:    for loading and running an open-weight model locally (Part 2)
# - accelerate:      lets transformers automatically place model weights on the GPU
# - huggingface_hub: for logging in to Hugging Face to download gated model weights
# - ipywidgets:      powers the optional model-picker in the bonus section
!pip install -q --upgrade google-genai transformers accelerate huggingface_hub ipywidgets

In [ ]:
import os
import time

import pandas as pd
import matplotlib.pyplot as plt
import torch

print("Libraries imported successfully.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

if device.type == "cuda":
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Part 2 (the local open-weight model) will still run on CPU, just much more slowly.")
    print("Go to Runtime -> Change runtime type -> GPU (T4), then re-run this cell.")

# We'll add one entry to this dictionary after each part -- exactly like a lab notebook --
# so we can compare both approaches side-by-side in Section 3.
results = {}

## Two ways to use a large language model

Every time you want an LLM to do something for you, you're really choosing between two fundamentally different arrangements:

| | **Closed-source API** (Part 1: Gemini) | **Open-weight, self-hosted** (Part 2: Gemma) |
|---|---|---|
| Where does it run? | On the provider's servers | On hardware you control (this Colab GPU, today) |
| What do you get access to? | An API endpoint — text in, text out | The actual model weights: files you can download, inspect, modify, fine-tune |
| Setup | An API key | Downloading multi-gigabyte weight files, picking a runtime, managing GPU memory |
| Cost model | Free tier, then pay-per-token | Free if you already have (or can borrow) a GPU; otherwise you pay for compute, not tokens |
| Data | Leaves your machine, goes to the provider | Never has to leave your machine |
| Capability | Usually the provider's most capable, most recently updated model | Whatever size you can fit on your hardware — typically smaller than the biggest closed models |
| Customization | Prompting (mostly) | Full fine-tuning, quantization, merging with other models |

Neither is "better" — they're suited to different jobs, and real production systems very often use **both**. Let's try each one, on the same prompt, and see for ourselves.

---
# 1. Part 1 — Calling a Closed-Source Model: the Gemini API

**Gemini** is Google's family of closed-source foundation models. The weights are never released — instead, Google runs the models on their own infrastructure and exposes them through an API. You authenticate with an API key, send a request, and a server somewhere runs the model and sends back a response.

This is the same pattern used by essentially every other major closed-source model provider, so what you learn here — authentication, the request/response shape, streaming, multi-turn state — transfers directly to other APIs.

### Step 1: Get a free Gemini API key

1. Go to **[Google AI Studio](https://aistudio.google.com/apikey)** and sign in with a Google account.
2. Click **Create API key**. AI Studio will automatically create a project for you if you don't already have one.
3. Copy the key — it starts with `AIza...`.

No credit card is required for the free tier. Free-tier requests are rate-limited (Flash models currently get a generous number of requests per minute, plus a daily cap — check the live numbers on the [rate limits page](https://ai.google.dev/gemini-api/docs/rate-limits), since these do change), which is more than enough for this session.

⚠️ **Note:** on the free tier, Google may use the prompts and responses you send to improve their products. Don't send anything sensitive through it.

### Step 2: Store the key securely using Colab Secrets

Never paste an API key directly into a notebook cell as text — it's easy to accidentally leak it if you share the notebook or push it to GitHub. Colab has a built-in **Secrets** manager for exactly this problem.

1. Click the **key icon (🔑)** in the left sidebar of Colab.
2. Click **+ Add new secret**.
3. Name it exactly `GEMINI_API_KEY` and paste your key as the value.
4. Toggle **Notebook access** on for this notebook.

The cell below reads it from there — your key never has to appear anywhere in this notebook's text.

In [ ]:
from google.colab import userdata

# Colab injects the secret into this runtime only -- it is never stored inside the .ipynb file itself.
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print("Gemini API key loaded from Colab secrets.")
except Exception as e:
    print("Could not load a secret named GEMINI_API_KEY.")
    print("Click the key icon on the left sidebar, add it, and make sure 'Notebook access' is turned on.")
    raise e

### Step 3: Install the SDK and create a client

We already installed `google-genai` in Section 0. The client automatically picks up the `GEMINI_API_KEY` environment variable we just set, so we don't need to pass it in explicitly.


In [ ]:
from google import genai

client = genai.Client()
print("Gemini client ready.")

# We can also use the "latest" alias rather than pinning an exact version string.
# Google swaps this alias to point at their newest stable Flash model, so this
# notebook keeps working -- and keeps using a current model -- even if you run it
# again months from now. If you'd rather pin an exact model for reproducibility,
# browse the full list at https://ai.google.dev/gemini-api/docs/models and use its
# endpoint string instead, e.g. "gemini-3.5-flash".

GEMINI_MODEL = "gemini-3.5-flash-lite"
# GEMINI_MODEL = "gemini-flash-latest"
print(f"Using model alias: {GEMINI_MODEL}")

### Step 4: Your first API call

Every call to the Interactions API follows the same shape: give it a `model` and some `input`, and get back an `Interaction` object. `interaction.output_text` is a convenience shortcut straight to the model's text response.


In [ ]:
# We'll reuse this exact prompt for the local open-weight model in Part 2,
# so our Section 3 comparison is measuring the same task both times.
SHARED_PROMPT = "Explain what a foundation model is, in exactly three sentences, for someone who has never studied computer science."

start_time = time.time()
interaction = client.interactions.create(
    model=GEMINI_MODEL,
    input=SHARED_PROMPT,
)
gemini_time = time.time() - start_time

print(interaction.output_text)
print(f"\n(Response took {gemini_time:.2f} seconds)")

### Step 5: Steering the model with a system instruction

A **system instruction** sets the model's persona and behavior for the whole conversation, kept separate from the user's actual question. This is how you'd give a production chatbot its personality, tone, or guardrails.


In [ ]:
interaction = client.interactions.create(
    model=GEMINI_MODEL,
    system_instruction="You are a pirate. Answer every question while staying fully in character.",
    input="What is a foundation model?",
)
print(interaction.output_text)

**Try it yourself:** change the system instruction above -- a Shakespearean tutor, an overly-formal professor, a five-year-old explaining it back to you -- and re-run the cell. Notice the *question* never changed, only the framing around it.

You can also control generation behavior, like `temperature` (higher = more random/creative, lower = more focused/deterministic), through `generation_config`:

[more on temperature](https://unstructured.io/insights/what-does-the-temperature-parameter-mean-in-llms)



In [ ]:
prompt = "Give me one creative use for a paperclip."

interaction = client.interactions.create(
    model=GEMINI_MODEL,
    input=prompt,
    generation_config={"temperature": 1.8},  # close to maximum randomness
)
print("High temperature (1.8):", interaction.output_text)

interaction = client.interactions.create(
    model=GEMINI_MODEL,
    input=prompt,
    generation_config={"temperature": 0.0},  # as deterministic as possible
)
print("\nLow temperature (0.0):", interaction.output_text)

### Step 6: Multi-turn conversations

The Interactions API keeps track of conversation history **on Google's servers** for you. Pass the previous interaction's `id` as `previous_interaction_id`, and the model has full context of everything said before -- you don't need to resend the whole conversation yourself.


In [ ]:
interaction1 = client.interactions.create(
    model=GEMINI_MODEL,
    input="I have 3 cats and 2 dogs at home.",
)
print("Turn 1:", interaction1.output_text)

interaction2 = client.interactions.create(
    model=GEMINI_MODEL,
    input="How many total legs do my pets have?",
    previous_interaction_id=interaction1.id,  # <- this is what makes it "remember" turn 1
)
print("\nTurn 2:", interaction2.output_text)

### Step 7: Streaming responses

So far we've waited for the *entire* response before printing anything. For anything user-facing, you usually want to stream text to the screen as it's generated -- the same effect you see in the Gemini app or ChatGPT.


In [ ]:
stream = client.interactions.create(
    model=GEMINI_MODEL,
    input="Count from 1 to 10, saying a brief word about why each number might be considered 'lucky' or 'unlucky' in different cultures.",
    stream=True,
)

for event in stream:
    if event.event_type == "step.delta" and event.delta.type == "text":
        print(event.delta.text, end="", flush=True)

### Step 8: Recording Gemini's results for later comparison

Let's log a clean, timed run on our shared prompt -- we'll do the exact same thing for the local model in Part 2.


In [ ]:
start_time = time.time()
interaction = client.interactions.create(model=GEMINI_MODEL, input=SHARED_PROMPT)
gemini_time = time.time() - start_time

# The Interactions API reports token usage on every response, so we don't have to
# estimate it ourselves the way we sometimes do with locally-run models.
output_tokens_gemini = interaction.usage.total_output_tokens if interaction.usage else None

print(interaction.output_text)

results["Gemini API (closed-source)"] = {
    "Runs on": "Google's servers",
    "Setup effort": "API key only",
    "Response time (s)": round(gemini_time, 2),
    "Output tokens": output_tokens_gemini,
    "Tokens/sec": round(output_tokens_gemini / gemini_time, 1) if output_tokens_gemini else None,
    "Cost (this session)": "$0 (free tier)",
    "Data leaves your machine?": "Yes",
}

print(f"\nLogged. Response time: {gemini_time:.2f}s, output tokens: {output_tokens_gemini}")

### Discussion: what did we just give up, and what did we get?

- We never once thought about GPUs, model weights, or memory -- Google handled all of it. What is that convenience actually costing you?
- Every prompt and response above traveled over the internet to Google's servers. When would that matter for something you're building?
- `gemini-flash-latest` will silently point at a newer model at some point in the future. When is that a feature, and when is it a bug?


---
# 2. Part 2 — Installing and Running an Open-Weight Model: Gemma 3

**Gemma** is Google's family of *open-weight* models -- built from the same research as Gemini, but with the weights published for anyone to download and run. Today we'll use **Gemma 3 1B** (`google/gemma-3-1b-it`), a small instruction-tuned model that comfortably fits on a free Colab T4 GPU.

"Open-weight" isn't quite the same thing as "open source": Google publishes the trained weights and lets you use, modify, and redistribute them under a license (the [Gemma Terms of Use](https://ai.google.dev/gemma/terms)), but the training data and training code aren't public. That's the common meaning of "open-weight" for most models you'll run into -- including Llama, Qwen, and Mistral.

### Step 1: Get access to Gemma on Hugging Face

Gemma is distributed through [Hugging Face](https://huggingface.co/), the most widely used hub for open-weight models, datasets, and demos. Because of its license, Gemma sits behind a one-click "gate":

1. Create a free account at [huggingface.co/join](https://huggingface.co/join) if you don't already have one.
2. Visit the model page: **[google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it)**.
3. Click **Acknowledge license** / **Agree and access repository**. This is processed instantly -- there's no waiting period or manual review.
4. Create an access token: **Settings → Access Tokens → Create new token** (a "Read" token is enough).

### Step 2: Store your Hugging Face token in Colab Secrets

Same pattern as the Gemini key:
1. Key icon (🔑) → **+ Add new secret**.
2. Name it `HF_TOKEN`, paste your access token as the value, turn on **Notebook access**.

In [ ]:
from huggingface_hub import login

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
except Exception as e:
    print("Could not load a secret named HF_TOKEN.")
    print("Add it via the key icon on the left sidebar (see Step 2 above), then re-run this cell.")
    raise e

### Step 3: Download and load the model

Unlike the API call in Part 1, this cell actually **downloads roughly 2 GB of model weights** to this Colab machine and loads them onto the GPU. This is the "installing an open-weight LLM" step -- once it finishes, the model runs entirely locally. No API key is needed to generate text after this point.

We load the weights as `torch.bfloat16` (16-bit floats) rather than the default 32-bit: this halves the memory footprint with a negligible quality difference, and it's standard practice for running LLMs on a single GPU. `device_map="auto"` (from the `accelerate` library we installed) automatically places the weights on the GPU if one is available.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GEMMA_MODEL_ID = "google/gemma-3-1b-it"

print(f"Downloading and loading {GEMMA_MODEL_ID}... (a minute or two the first time)")

gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_ID)
gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
gemma_model.eval()  # inference mode: disables dropout and other training-only behavior

n_params_gemma = sum(p.numel() for p in gemma_model.parameters())
print(f"\nModel loaded. Parameters: {n_params_gemma:,} (~{n_params_gemma / 1e9:.2f}B)")
print(f"Model device: {gemma_model.device}")

In [ ]:
# A quick, practical check: how much of the T4's 16GB of VRAM are we actually using?
# Handy to know before you try loading a second, bigger model in the bonus section.
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

### Step 4: Write a prompt using the chat template

Instruction-tuned models like Gemma expect input in a specific chat format -- a list of turns, each with a `role` (`system`, `user`, or `model`/`assistant`) and plain-text `content`. Every chat model ships with its own **chat template**: a small piece of logic (bundled with the tokenizer) that turns that list of turns into the exact text format the model was fine-tuned on, delimiters and all.

`tokenizer.apply_chat_template(...)` builds that formatted prompt for us; `model.generate(...)` then runs it through the model. We only decode the *newly generated* tokens, so the printed output is just the model's reply, not our prompt echoed back.


In [ ]:
def ask_model(model, tokenizer, prompt, system_prompt="You are a helpful, concise assistant.", max_new_tokens=200, temperature=0.7):
    '''Send one prompt to a locally-loaded chat model and return its reply as plain text.
    Works for Gemma and for almost any other instruction-tuned causal language model on
    Hugging Face (we reuse this exact function for the bonus section in Part 4).
    '''

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-4),  # 0.0 isn't a valid sampling temperature
        )

    # output_ids includes our prompt followed by the new tokens -- slice off the prompt
    # so we only decode what the model actually generated.
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


response = ask_model(gemma_model, gemma_tokenizer, SHARED_PROMPT)
print(response)

### Step 5: Recording Gemma's results for later comparison

Just like Part 1, let's log a timed run on the exact same `SHARED_PROMPT`.


In [ ]:
start_time = time.time()
response_text = ask_model(gemma_model, gemma_tokenizer, SHARED_PROMPT)
gemma_time = time.time() - start_time

output_tokens_gemma = len(gemma_tokenizer(response_text)["input_ids"])

print(response_text)

results["Gemma 3 1B (open-weight, local)"] = {
    "Runs on": str(device).upper(),
    "Setup effort": "HF account + license + ~2GB download",
    "Response time (s)": round(gemma_time, 2),
    "Output tokens": output_tokens_gemma,
    "Tokens/sec": round(output_tokens_gemma / gemma_time, 1),
    "Cost (this session)": "$0 (free Colab GPU)",
    "Data leaves your machine?": "No",
}

print(f"\nLogged. Response time: {gemma_time:.2f}s, output tokens: {output_tokens_gemma}")

**Try it yourself:** call `ask_model(gemma_model, gemma_tokenizer, "your prompt here", system_prompt="...")` with a different question or persona -- same idea as the Gemini system instruction in Part 1, just running on this machine instead of Google's.

### Discussion: what did we just give up, and what did we get?

- Gemma 3 1B is *far* smaller than Gemini's production models (roughly one billion parameters, versus an undisclosed but much larger number). What did you notice about response quality compared to Part 1?
- Once loaded, `ask_model` needs no internet connection at all to keep generating text. When would that matter?
- If you wanted to fine-tune a model on your own company's data, could you do that with the Gemini API? Could you do it here?


---
# 3. Head-to-Head Comparison

We've now solved the exact same task two different ways. Let's put the results side by side.


In [ ]:
summary_df = pd.DataFrame(results).T
summary_df.index.name = "Method"
summary_df

In [ ]:
methods = list(results.keys())
tokens_per_sec = [results[m]["Tokens/sec"] for m in methods]

plt.figure(figsize=(9, 4.5))
bars = plt.bar(methods, tokens_per_sec, color=["#4C72B0", "#55A868"])
plt.ylabel("Tokens generated per second")
plt.title("Generation Speed: Closed API vs. Local Open-Weight Model")
for bar, val in zip(bars, tokens_per_sec):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{val:.1f}", ha="center", va="bottom")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Discussion

- Which approach was faster on this prompt? Does that surprise you? (Hint: think about what "1 billion parameters running on a dedicated free GPU" is competing against on the API side -- network round-trip time, queueing behind other users, and a much larger model doing the answering.)
- Neither "cost" row is really `$0` in a deeper sense. What is each approach actually costing you, even on the free tier?
- Imagine you're building a product that needs to run on a phone with no internet connection. Which approach is even possible?
- Imagine you're building a product that needs the best possible answer quality, and cost genuinely isn't a factor. Which approach would you reach for?
- Real systems often use **both**: a large closed-source API for hard or rare questions, and a small open-weight model for cheap, high-volume, or offline tasks. Can you think of a product that might want to split work this way?


---
# 4. Bonus -- Swap in a Different Open-Weight Model

*This section is optional.*

Gemma isn't the only open-weight option, and not every model sits behind a license gate the way Gemma does. Pick another small instruction-tuned model below and see how it compares -- the exact same `ask_model` helper you just used works for almost any chat model on Hugging Face.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

model_dropdown = widgets.Dropdown(
    options=[
        ("Qwen2.5 1.5B Instruct  --  Alibaba, ~1.5B params, ungated", "Qwen/Qwen2.5-1.5B-Instruct"),
        ("SmolLM2 1.7B Instruct  --  Hugging Face, ~1.7B params, ungated", "HuggingFaceTB/SmolLM2-1.7B-Instruct"),
        ("Gemma 3 1B (again, as a sanity check)", "google/gemma-3-1b-it"),
    ],
    description="Pick a model:",
    style={"description_width": "initial"},
)
display(model_dropdown)

In [ ]:
SECOND_MODEL_ID = model_dropdown.value
print(f"Downloading and loading {SECOND_MODEL_ID}...")

second_tokenizer = AutoTokenizer.from_pretrained(SECOND_MODEL_ID)
second_model = AutoModelForCausalLM.from_pretrained(
    SECOND_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
second_model.eval()

n_params_second = sum(p.numel() for p in second_model.parameters())
print(f"Model loaded. Parameters: {n_params_second:,} (~{n_params_second / 1e9:.2f}B)")

start_time = time.time()
second_response = ask_model(second_model, second_tokenizer, SHARED_PROMPT)
second_time = time.time() - start_time

print(f"\n{second_response}")
print(f"\n({SECOND_MODEL_ID} took {second_time:.2f}s on the shared prompt)")

**Want to compare more than two open models at once?** Re-run the two cells above with a different dropdown selection, storing each result under its own key in `results` (e.g. `results[SECOND_MODEL_ID] = {...}`, following the same pattern as Section 2, Step 5) -- Section 3's table and chart will then show every model you've tried, side by side.

In [ ]:
output_tokens_second = len(second_tokenizer(second_response)["input_ids"])

results[SECOND_MODEL_ID] = {
    "Runs on": str(device).upper(),
    "Setup effort": "HF account + license + ~3GB download",
    "Response time (s)": round(second_time, 2),
    "Output tokens": output_tokens_second,
    "Tokens/sec": round(output_tokens_second / second_time, 1),
    "Cost (this session)": "$0 (free Colab GPU)",
    "Data leaves your machine?": "No",
}

A natural next question is: *what if the model you want doesn't fit in this GPU's memory at all?* The short answer is **quantization** -- loading the weights at lower precision (8-bit or 4-bit instead of 16-bit) via the `bitsandbytes` library, trading a small amount of quality for a large drop in memory use. That's how people run 7B-13B parameter open-weight models on the exact same free T4 GPU. To explore: see the [Transformers quantization guide](https://huggingface.co/docs/transformers/main/en/quantization/overview).

---
# Wrap-Up

Today we used one large language model capability two structurally different ways:

- **Gemini API (closed-source)** -- no setup beyond an API key, always the provider's latest model, but your data leaves your machine and you pay per token past the free tier.
- **Gemma 3, running locally (open-weight)** -- real setup effort (accounts, licenses, downloads, GPU memory), but the model is now genuinely yours: offline-capable, inspectable, and fine-tunable, at zero marginal cost per request.

Most real products end up using a mix of both, matched to the job: a capable closed API for the hard, rare, or high-stakes questions, and a cheap local or open-weight model for everything high-volume, latency-sensitive, or privacy-sensitive.

### Where to go from here
- [Gemini API docs](https://ai.google.dev/gemini-api/docs) -- structured output, function calling, and Gemini's other tools (Google Search grounding, code execution, image/video/audio understanding)
- [Hugging Face course](https://huggingface.co/course) -- a free, thorough introduction to the wider open-weight ecosystem
- [Gemma model card](https://huggingface.co/google/gemma-3-1b-it) -- other sizes (270M up to 27B) and the multimodal 4B+ variants
- [Transformers quantization guide](https://huggingface.co/docs/transformers/main/en/quantization/overview) -- how to fit bigger open-weight models on the same free GPU
- [Hugging Face model fine-tuning guides for Gemma](https://ai.google.dev/gemma/docs/core/huggingface_text_full_finetune) -- the natural next step once you've got a model loaded locally


<!--
[Your Name]
[Date]
-->
<br>

---

Instructor: **Rochana R. Obadage**<br>
Last Updated: _08th September 2026_